# Flipkart Product Review Sentiment

Real Flipkart reviews this time. I load `ml-hub/flipkart-reviews` from Hugging Face, a public copy of reviews scraped from Flipkart product pages (air coolers, phones, appliances and so on). It ships with a 30,000-row train split and a 10,000-row test split, and each review has a star rating, a short title, the review text, and a positive / neutral / negative sentiment label. The dataset card does not say how the labels were made, and they do not map one-to-one to stars (some 3-star reviews are labeled positive, others neutral).

I keep all three classes and use the provided train/test split as is. Model is a TF-IDF + LinearSVC baseline on title plus body.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

np.random.seed(7)
DATA_DIR = Path("../../data/flipkart_sentiment")


In [2]:
BASE = "https://huggingface.co/datasets/ml-hub/flipkart-reviews/resolve/main/data/"
DATA_DIR.mkdir(parents=True, exist_ok=True)
for split in ["train", "test"]:
    path = DATA_DIR / f"{split}.parquet"
    if not path.exists():
        pd.read_parquet(BASE + f"{split}-00000-of-00001.parquet").to_parquet(path)

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
for d in (train_df, test_df):
    d["review"] = (d["Review"].fillna("") + ". " + d["text"].fillna("")).str.strip()
    d["sentiment"] = d["labels"]
print(train_df.shape, test_df.shape)
print(train_df.sentiment.value_counts())
train_df[["Rate", "review", "sentiment"]].head()

(30000, 9) (10000, 9)
sentiment
positive    24287
negative     4335
neutral      1378
Name: count, dtype: int64


,Rate,review,sentiment
0,5.0,super!. great cooler excellent air flow and fo...,positive
1,5.0,awesome. best budget 2 fit cooler nice cooling,positive
2,3.0,fair. the quality is good but the power of air...,positive
3,1.0,useless product. very bad product its a only a...,negative
4,3.0,fair. ok ok product,neutral


## TF-IDF baseline + LinearSVC

In [3]:
X_train, y_train = train_df["review"].values, train_df["sentiment"].values
X_test, y_test = test_df["review"].values, test_df["sentiment"].values

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

# LinearSVC because the text is short; class_weight="balanced" because neutral is the small class
clf = LinearSVC(class_weight="balanced")
clf.fit(Xtr, y_train)
pred = clf.predict(Xte)

print(classification_report(y_test, pred, digits=3))

              precision    recall  f1-score   support

    negative      0.870     0.935     0.901      1347
     neutral      0.593     0.617     0.605       347
    positive      0.988     0.974     0.981      8306

    accuracy                          0.956     10000
   macro avg      0.817     0.842     0.829     10000
weighted avg      0.958     0.956     0.957     10000



## Notes

On the 10,000-review test split the TF-IDF + LinearSVC baseline gets 95.6% accuracy and 0.829 macro-F1. Accuracy flatters it because 83% of reviews are positive. The per-class numbers are the real story:

- Positive is easy (F1 0.98) and negative is solid (F1 0.90).
- Neutral is the weak spot at F1 0.61. It's only 1,378 of the 30,000 training reviews, and phrases like "ok ok product" or "good but the air flow is average" sit right between the other two classes.

The failure modes I'd expect to matter most on this data:
- Mixed-language reviews (Hinglish), where TF-IDF tokenization gets weird.
- Sarcasm and qualifiers ("works great after I returned it twice").
- Star-rating inconsistency. People sometimes give 5 stars with a negative-sounding review, and the labels here don't line up perfectly with stars either.

For a production version I'd try a fine-tuned multilingual model (XLM-R or MuRIL for Indic content) and spend most of the effort on the neutral class.